# Практика · Робота з API

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)

**Мережа не потрібна:** усі запити йдуть на наш власний сервер, який зошит піднімає
сам на твоєму компʼютері. Це не заглушка — `requests` відкриває справжнє зʼєднання
й отримує справжні коди станів. Відрізняється лише адреса: замість чужого домену
там `127.0.0.1`.

Наскрізний приклад той самий, що в лекції та в темі 22, — маленька кавʼярня.
Тільки тепер її меню живе не у файлі, а за адресою `/menu`.

Що зробимо:

1. піднімемо навчальний API кавʼярні й переконаємось, що він відповідає;
2. розберемо відповідь через `r.json()` і звіримо результат із `json.loads()`;
3. спробуємо `r.json()` на сторінці помилки й побачимо `JSONDecodeError`;
4. навчимося перевіряти `Content-Type` замість того, щоб сподіватися;
5. створимо позицію меню через `POST`, прочитаємо код `201` і заголовок `Location`;
6. отримаємо `422` без обовʼязкового поля й прочитаємо тіло помилки;
7. побачимо на власні очі різницю між `json=` і `data=`;
8. пройдемо всі три коди автентифікації: `401`, `403`, `200`;
9. обійдемо пагінацію циклом «поки є `next`» зі стелею кроків;
10. отримаємо `429` разом із `Retry-After`;
11. напишемо повтор із зростаючою паузою для маршруту, що двічі відповідає `503`;
12. зупинимо сервер.

## 1 · Риштування: навчальний сервер

Наступна клітинка — **риштування**, а не матеріал теми. Вона описує наш API
через **клас**, а класи будуть аж у [темі 30](../30-classes/lecture.html).
Зараз її не треба розуміти й тим паче запамʼятовувати — просто запусти.

Що варто знати про неї трьома реченнями: сервер слухає на `127.0.0.1` (це адреса
твого ж компʼютера), порт `0` означає «візьми будь-який вільний», а працює він
у фоновому потоці, тож зошит не зависає. Усі маршрути, які нам знадобляться,
описані всередині: меню зі сторінками, закритий розділ із токеном, маршрут із
лімітом і маршрут, який спершу двічі ламається.

In [ ]:
import json
import os
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import requests

# меню кавʼярні — те саме, що в темі 22, тільки тепер воно живе на сервері
MENU = [
    {"id": 1, "name": "Еспресо",  "price": 25, "category": "кава"},
    {"id": 2, "name": "Капучино", "price": 45, "category": "кава"},
    {"id": 3, "name": "Латте",    "price": 50, "category": "кава"},
    {"id": 4, "name": "Раф",      "price": 55, "category": "кава"},
    {"id": 5, "name": "Чай",      "price": 30, "category": "чай"},
]
PAGE_SIZE = 2                 # сервер віддає меню по дві позиції за раз
TOKEN = "demo-token"          # ключ до закритого розділу
state = {"flaky_calls": 0}    # лічильник для маршруту, що спершу ламається


class CafeAPI(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"      # без цього keep-alive не працює
    disable_nagle_algorithm = True     # без цього відповіді чекають зайві 40 мс

    def log_message(self, *args):
        pass                           # інакше зошит заросте рядками журналу

    def reply(self, code, payload, headers=None):
        """Віддає словник як JSON із потрібними заголовками."""
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        for name, value in (headers or {}).items():
            self.send_header(name, value)
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        path, _, query = self.path.partition("?")
        params = dict(p.split("=", 1) for p in query.split("&") if "=" in p)

        if path == "/menu":
            page = int(params.get("page", 1))
            start = (page - 1) * PAGE_SIZE
            items = MENU[start:start + PAGE_SIZE]
            has_more = start + PAGE_SIZE < len(MENU)
            self.reply(200, {"page": page, "total": len(MENU),
                             "next": page + 1 if has_more else None,
                             "items": items})

        elif path == "/secret":
            auth = self.headers.get("Authorization", "")
            if not auth:
                self.reply(401, {"error": "потрібен Authorization"})
            elif auth != f"Bearer {TOKEN}":
                self.reply(403, {"error": "токен без доступу"})
            else:
                self.reply(200, {"revenue": 12450, "orders": 217})

        elif path == "/limited":
            self.reply(429, {"error": "забагато запитів"}, {"Retry-After": "2"})

        elif path == "/flaky":
            state["flaky_calls"] += 1
            if state["flaky_calls"] < 3:
                self.reply(503, {"error": "сервіс тимчасово недоступний"})
            else:
                self.reply(200, {"ok": True, "attempt": state["flaky_calls"]})

        elif path == "/menu.html":
            # та сама сторінка меню, але для очей — знадобиться для пастки
            page = "<html><body><h1>Меню</h1><p>Еспресо — 25 грн</p></body></html>"
            body = page.encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        else:
            self.reply(404, {"error": "немає такого маршруту"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length)

        if self.path == "/echo":
            # дзеркало: повертає рівно те, що отримав, — щоб побачити тіло запиту
            self.reply(200, {"content_type": self.headers.get("Content-Type", ""),
                             "body": raw.decode("utf-8")})
            return

        if self.path != "/menu":
            self.reply(404, {"error": "немає такого маршруту"})
            return

        try:
            data = json.loads(raw.decode("utf-8"))
        except json.JSONDecodeError:
            self.reply(400, {"error": "тіло не є JSON"})
            return

        missing = [field for field in ("name", "price") if field not in data]
        if missing:
            self.reply(422, {"error": "бракує обовʼязкових полів", "missing": missing})
            return

        created = {"id": 99, "name": data["name"], "price": data["price"],
                   "category": data.get("category", "інше")}
        self.reply(201, created, {"Location": "/menu/99"})


server = ThreadingHTTPServer(("127.0.0.1", 0), CafeAPI)
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_address[1]}"

print("наш API працює за адресою:", BASE)
print("це адреса твого ж компʼютера — у мережу нічого не виходить")

## 2 · Перший запит: що взагалі прийшло

Перед тим як щось розбирати, подивимось на відповідь цілком: код стану, заголовок
`Content-Type` і сирий текст. Це три речі, з яких починається будь-яка розмова
з незнайомим API.

In [ ]:
response = requests.get(f"{BASE}/menu", params={"page": 1}, timeout=5)

print("код стану:  ", response.status_code)
print("Content-Type:", response.headers["Content-Type"])
print("адреса:     ", response.url)          # params= вбудувались у неї самі
print()
print("сирий текст відповіді:")
print(response.text)

## 3 · `r.json()`: з тексту в обʼєкти Python

Текст вище — це JSON. Метод `r.json()` розбирає його й повертає звичайні обʼєкти
Python. Щоб переконатися, що жодної магії там немає, зробимо те саме вручну через
`json.loads()` з [теми 22](../22-csv-json/lecture.html) і звіримо результати.

In [ ]:
data = response.json()                    # спосіб від requests
manual = json.loads(response.text)        # той самий розбір, зроблений руками

print("тип того, що повернув r.json():", type(data))
print("ключі відповіді:", list(data.keys()))
print("позицій на сторінці:", len(data["items"]))
print("перша позиція:", data["items"][0])

# найцінніша перевірка практики: всередині бібліотеки немає магії
assert data == manual, "r.json() і json.loads() дали різне!"
print("\n✅ r.json() робить рівно те саме, що json.loads(r.text)")

## 4 · Чому це краще за текст

З HTML ціна прийшла б рядком `"25 грн"`. З JSON вона приходить числом — і з нею
одразу можна рахувати. Перевіримо це не на віру, а через `type()` і додавання.

In [ ]:
price = data["items"][0]["price"]

print("значення:", price)
print("тип:     ", type(price))
print("ціна з націнкою 5 грн:", price + 5)     # з рядком це впало б із TypeError

assert isinstance(price, int), "ціна мала прийти числом!"
print("\n✅ перетворювати нічого не треба — число вже число")

## 5 · Пастка: `r.json()` на не-JSON

У нашого сервера є ще одна адреса — `/menu.html`. Там лежить те саме меню, але
у вигляді сторінки для очей. Сервер відповідає **успішно** (код `200`), тож
`raise_for_status()` промовчить. А `r.json()` впаде.

Саме так це виглядає в житті: щось пішло не так, і замість даних прилетіла
сторінка помилки або сторінка входу.

In [ ]:
page_response = requests.get(f"{BASE}/menu.html", timeout=5)

print("код стану:  ", page_response.status_code, "— тобто успіх")
print("Content-Type:", page_response.headers["Content-Type"])
print("тіло:", page_response.text)
print()

try:
    page_response.json()
except requests.exceptions.JSONDecodeError as error:
    # розбирач стояв на першому символі й чекав початку значення JSON,
    # а побачив «<» — звідси й дивне на вигляд повідомлення
    print("виняток:", type(error).__name__)
    print("повідомлення:", error)

## 6 · Лікування: питай `Content-Type`, а не сподівайся

Сервер сам каже, що надіслав. Напишемо маленьку функцію, яка розбирає відповідь
лише тоді, коли це справді JSON, і чесно повідомляє про проблему в іншому разі.

In [ ]:
def parse_json_safely(response):
    """Повертає розібране тіло або None, якщо сервер надіслав не JSON."""
    kind = response.headers.get("Content-Type", "")
    if not kind.startswith("application/json"):
        print(f"  ⚠️ очікували JSON, а прийшло {kind!r}")
        print(f"     початок тіла: {response.text[:60]!r}")
        return None
    return response.json()


print("на /menu:")
good = parse_json_safely(requests.get(f"{BASE}/menu", timeout=5))
print("  розібрано, позицій:", len(good["items"]))

print("\nна /menu.html:")
bad = parse_json_safely(requests.get(f"{BASE}/menu.html", timeout=5))
print("  результат:", bad)

assert good is not None and bad is None, "перевірка Content-Type не спрацювала!"
print("\n✅ на не-JSON функція не падає, а пояснює, що сталося")

## 7 · Створюємо позицію меню: `POST`, `201`, `Location`

Тепер не читаємо, а пишемо. Метод `POST`, адреса **колекції** `/menu`, нові дані —
у тілі запиту через аргумент `json=`.

Дивись на три речі: код `201` (а не `200`), заголовок `Location` з адресою
створеного ресурсу й `id`, який вигадав сервер, а не ми.

In [ ]:
created = requests.post(f"{BASE}/menu",
                        json={"name": "Какао", "price": 40, "category": "какао"},
                        timeout=5)

print("код стану:", created.status_code)
print("Location: ", created.headers["Location"])
print("тіло:     ", created.json())
print()
print("Content-Type, який requests поставив САМ:",
      created.request.headers["Content-Type"])

assert created.status_code == 201, "створення мало дати 201"
assert created.headers["Location"] == "/menu/99"
assert created.json()["id"] == 99, "номер нового запису вигадує сервер"
print("\n✅ ресурс створено, його адреса — у заголовку Location")

## 8 · Коли сервер каже «ні»: код `422` і тіло помилки

Спробуємо створити позицію без ціни. Сервер відповість `422`: тіло є, це справжній
JSON, але обовʼязкового поля бракує.

Головне тут — **прочитати тіло**. Дивитись лише на код і зупинятись — найчастіша
помилка новачка: сервер часто пише, чого саме бракує.

In [ ]:
rejected = requests.post(f"{BASE}/menu", json={"name": "Какао"}, timeout=5)

print("код стану:", rejected.status_code)
print("тіло помилки:", rejected.json())
print("сервер прямо назвав поле:", rejected.json()["missing"])

# 4xx означає «виправ свій запит»: повторювати без змін безглуздо
again = requests.post(f"{BASE}/menu", json={"name": "Какао"}, timeout=5)
print("\nповторили той самий запит — код:", again.status_code, "(і буде так завжди)")

assert rejected.status_code == 422
assert rejected.json()["missing"] == ["price"]
print("\n✅ на 4xx лікують запит, а не чекають")

## 9 · `json=` проти `data=`: різниця, яку не видно

Обидва аргументи кладуть словник у тіло запиту — але **по-різному**, і ставлять
різний `Content-Type`. Плутанина між ними дає `400` на цілком правильних даних.

Щоб побачити це на власні очі, у нашого сервера є маршрут-дзеркало `/echo`:
він повертає рівно те, що отримав.

In [ ]:
payload = {"name": "Какао", "price": 40}

as_json = requests.post(f"{BASE}/echo", json=payload, timeout=5).json()
as_form = requests.post(f"{BASE}/echo", data=payload, timeout=5).json()

print("json=  →", as_json["content_type"])  # \u0410-послідовності — це теж JSON
print("        тіло:", as_json["body"])
print()
print("data=  →", as_form["content_type"])
print("        тіло:", as_form["body"])

# кирилиця в тілі виглядає як \u041a…: requests екранує її, і це нормальний JSON —
# приймальна сторона розбере його в ті самі «Какао»
assert json.loads(as_json["body"]) == payload
assert as_json["content_type"] == "application/json"
assert as_form["content_type"] == "application/x-www-form-urlencoded"
print("\n✅ json= надсилає JSON, data= — поля форми; заголовок requests ставить сам")

## 10 · Автентифікація: `401`, `403` і `200`

Закритий розділ `/secret` віддає виторг. Пройдемо всі три стани й побачимо
різницю між «я не знаю, хто ти» (`401`) і «знаю, але тобі не можна» (`403`).

Токен беремо зі **змінної оточення**, а не пишемо в код. Тут ми задамо цю змінну
самі, щоб зошит працював у будь-кого, — але в справжньому проєкті її ставлять
ззовні, а файл `.env` вписують у `.gitignore`.

In [ ]:
# у справжньому проєкті цей рядок зайвий: змінну задають ззовні, перед запуском
os.environ.setdefault("CAFE_TOKEN", "demo-token")

token = os.environ["CAFE_TOKEN"]          # ключ живе поза кодом

attempts = {
    "без заголовка":  {},
    "хибний токен":   {"Authorization": "Bearer wrong-token"},
    "чинний токен":   {"Authorization": f"Bearer {token}"},
}

codes = {}
for label, headers in attempts.items():
    answer = requests.get(f"{BASE}/secret", headers=headers, timeout=5)
    codes[label] = answer.status_code
    print(f"{label:<15} → {answer.status_code}  {answer.json()}")

assert codes["без заголовка"] == 401, "без ключа сервер має відповісти 401"
assert codes["хибний токен"] == 403, "з чужим ключем — 403"
assert codes["чинний токен"] == 200
print("\n✅ 401 — «представся», 403 — «прав бракує»; це різні хвороби")

## 11 · Пагінація: цикл «поки є `next`»

Сервер віддає меню по дві позиції за раз. Зберемо всі пʼять, ідучи за полем
`next`, доки воно не стане `None`.

Зверни увагу на лічильник `requests_made`: це **стеля кроків**. Умову виходу
з цього циклу визначає чужа програма, тож без стелі помилка на боці сервісу
зациклила б нас назавжди.

In [ ]:
all_items = []
page = 1
requests_made = 0
LIMIT = 20                       # стеля: більше сторінок ми не чекаємо

while page is not None:
    if requests_made >= LIMIT:
        print("забагато сторінок, зупиняюсь")
        break

    answer = requests.get(f"{BASE}/menu", params={"page": page}, timeout=5)
    answer.raise_for_status()
    requests_made += 1

    body = answer.json()
    all_items.extend(body["items"])
    print(f"сторінка {body['page']}: позицій {len(body['items'])}, next = {body['next']}")

    page = body["next"]          # None на останній сторінці

print(f"\nзроблено запитів: {requests_made}, зібрано позицій: {len(all_items)}")
print("назви:", [item["name"] for item in all_items])

assert requests_made == 3, "пʼять позицій по дві на сторінку — це три запити"
assert len(all_items) == 5
assert [item["name"] for item in all_items] == [m["name"] for m in MENU]
print("\n✅ зібрали рівно те саме меню, що лежить на сервері")

## 12 · Ліміт: код `429` і заголовок `Retry-After`

Маршрут `/limited` завжди відповідає `429` — так поводиться API, у якого ти
вичерпав дозволену частоту. Разом із кодом він надсилає `Retry-After`: скільки
секунд чекати.

Це не порада, а вказівка. Число дає сервер — він краще знає, коли твій ліміт
відновиться.

In [ ]:
limited = requests.get(f"{BASE}/limited", timeout=5)

print("код стану:  ", limited.status_code)
print("Retry-After:", limited.headers["Retry-After"], "(секунд)")
print("тіло:       ", limited.json())

wait_seconds = float(limited.headers["Retry-After"])
print(f"\nу справжній програмі тут стояло б time.sleep({wait_seconds})")
print("у зошиті чекати по-справжньому не будемо — просто памʼятай, що так треба")

assert limited.status_code == 429
assert wait_seconds == 2.0
print("\n✅ сервер сам сказав, скільки чекати")

## 13 · Повтори з відступом: `503`, `503`, `200`

Маршрут `/flaky` двічі відповідає `503`, а на третій спробі — `200`. Це
поведінка сервісу, який перезапускається: код `5xx` означає «зараз не можу»,
і слово «зараз» тут ключове.

Напишемо повтор із **зростаючою** паузою. У справжній програмі паузу беруть
секундами (1, 2, 4, 8); тут ми стартуємо з `0.02` с, щоб зошит не спав хвилинами.
Пропорція між паузами від цього не міняється — а саме вона і є суттю.

In [ ]:
def get_with_retry(url, attempts=5, pause=0.02):
    """Повторює запит, поки сервер віддає 5xx. Пауза щоразу подвоюється."""
    log = []
    for number in range(1, attempts + 1):
        answer = requests.get(url, timeout=5)

        if answer.status_code < 500:
            log.append((number, answer.status_code, None))
            return answer, log        # 2xx або 4xx — повторювати нічого

        log.append((number, answer.status_code, round(pause, 3)))
        time.sleep(pause)
        pause = pause * 2             # ось він, відступ

    return answer, log


answer, log = get_with_retry(f"{BASE}/flaky")

for number, status, waited in log:
    if waited is None:
        print(f"спроба {number}: код {status} — готово, чекати нема чого")
    else:
        print(f"спроба {number}: код {status} — чекаємо {waited} с і пробуємо знову")

print("\nвідповідь сервера:", answer.json())

assert [row[1] for row in log] == [503, 503, 200], "мало бути дві невдачі й успіх"
assert [row[2] for row in log] == [0.02, 0.04, None], "пауза мала подвоюватись"
print("\n✅ третя спроба дочекалась 200, а пауза між спробами зростала вдвічі")

Чому пауза саме **зростає**, а не лишається однаковою? Порахуймо словами: пʼять
спроб із подвоєнням (1, 2, 4, 8 с) покривають майже шістнадцять секунд збою,
а пʼять спроб по секунді — лише чотири. І головне: якщо тисяча клієнтів
одночасно грюкатиме раз на секунду, перевантажений сервер не встане ніколи.

І ще одне правило, яке легко забути: повторювати можна не всі методи. `GET`,
`PUT` і `DELETE` ідемпотентні. А `POST` створює новий запис щоразу — його повтор
після обірваного зʼєднання дав би два замовлення замість одного.

## 14 · Звичка, яка економить пів години

Знайомлячись із новим API, надрукуй відповідь красиво **перед** тим, як писати
шляхи до полів. Два аргументи роблять усю роботу: `indent=2` розкладає структуру
по рядках, `ensure_ascii=False` лишає кирилицю кирилицею.

In [ ]:
sample = requests.get(f"{BASE}/menu", params={"page": 2}, timeout=5).json()

print(json.dumps(sample, indent=2, ensure_ascii=False))
print()
print('шлях до ціни другої позиції:  data["items"][1]["price"] →',
      sample["items"][1]["price"])
print('безпечніше на чужих даних:    data.get("next") →', sample.get("next"))
print('а на неіснуючому ключі:       data.get("total_pages") →',
      sample.get("total_pages"))

## 15 · Зупиняємо сервер

Сервер жив у фоновому потоці весь цей час. Приберемо за собою — інакше він
триматиме порт до кінця роботи ядра.

In [ ]:
server.shutdown()
server.server_close()

print("сервер зупинено, порт звільнено")
print("всього маршрутів ми зачепили: /menu, /menu.html, /secret, /limited, /flaky, /echo")

## Завдання трьох рівнів

Повний текст із критеріями «зроблено» — у [homework.html](homework.html).
Коротко, щоб було видно напрямок:

🟢 **База.** Підніми сервер знову (перезапусти клітинку 1) і напиши функцію
`fetch_all(base_url)`, яка повертає список усіх позицій меню. Усередині —
цикл по `next` зі стелею кроків. Перевір `assert`-ом, що позицій пʼять.

🟡 **Плюс.** Додай до сервера маршрут `GET /menu/<id>`: він має віддавати одну
позицію або `404`, якщо такої немає. Напиши функцію, яка бере `id` і повертає
назву позиції або `None`, і покажи, що на `id = 999` вона не падає.

🔴 **Виклик.** Зроби ліміт частоти справжнім: хай сервер рахує запити й після
пʼятого за секунду віддає `429` з `Retry-After`. Далі напиши клієнта, який
обходить усе меню, жодного разу не отримавши `429`, і доведи це лічильником.